In [ ]:
import os
import sys
from pathlib import Path
import sys, time
import subprocess
import time
import requests
from dotenv import load_dotenv
from openai import OpenAI
from langchain_ollama import ChatOllama
from langchain_openai import ChatOpenAI
from langchain_anthropic import ChatAnthropic

# Make aai/ importable from ipy_nb/
sys.path.insert(0, "..")
from aai.prompts import (
    FILE_SUMMARIZER_PROMPT,
    CONTEXT_MANAGER_PROMPT,
    ARCHITECT_PROMPT,
)
from aai.repo_reader import TEXT_SUFFIXES

# --- CONFIGURATION ---
REPO_PATH = "../code_base"
OUTPUT_DIR = Path("output_analysis")
MAX_CHUNK_CHARS = 12000 # Roughly 3k-4k tokens

# Initialize Directories
STAGES = ["01_scout", "02_aggregate", "03_draft", "04_critique", "05_refined", "06_visual"]
for stage in STAGES:
    (OUTPUT_DIR / stage).mkdir(parents=True, exist_ok=True)

# Load environment variables
load_dotenv()

# TODO
changes I want to make....

Integrate with Manikandan's prompts + verify the aggregator.
The LLM server is using a local llm instead of anthropics. I'll change this, and reference the .env file. 
The agents should extend a base class's functionality. I think an llm would refactor it pretty quickly...
More refactoring... Use document chunking and crawling from langchain. Lower priority...
Move the modules to a better file structure/use Manikandan's code base. This one should have some discussion, but I kind of like the idea of using classes because it's more stateful imo.

In [7]:
# list ollama models
!ollama list

In [ ]:
# list cached models (on mac hardware)
!ls -F ~/.cache/huggingface/hub/

In [ ]:
# For macs, mlx-lm is more optimized on the hardware
# Default running a 4 bit quantized version
def start_mlx_server(model_id="mlx-community/Qwen2.5-7B-Instruct-4bit", port=8080):
    # 1. Check if server is already running
    try:
        requests.get(f"http://localhost:{port}/v1/models", timeout=2)
        print(f"Server already running on port {port}.")
        return None
    except:
        print(f"Starting MLX server with {model_id}...")
        # 2. Start the process
        cmd = ["python", "-m", "mlx_lm.server", "--model", model_id, "--port", str(port)]
        process = subprocess.Popen(cmd, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
        
        # 3. Wait for readiness
        for _ in range(30):
            try:
                requests.get(f"http://localhost:{port}/v1/models", timeout=1)
                print("Server is ready!")
                return process
            except:
                time.sleep(2)
        print("Failed to start server.")
        return None


def get_model():
    """Returns a LangChain chat model based on LLM_PROVIDER env var.
    
    Supported providers (set LLM_PROVIDER in .env):
      - 'local'        : MLX local server via ChatOpenAI-compatible wrapper
      - 'local-ollama' : Ollama via ChatOllama
      - 'claude'       : Anthropic Claude Sonnet via ChatAnthropic
      - 'openai'       : OpenAI GPT-4o via ChatOpenAI (default)
    """
    provider = os.getenv("LLM_PROVIDER", "local").lower()

    if provider == "local":
        # Ensure MLX server is running: python -m mlx_lm.server --model <model_id>
        start_mlx_server()
        model_id = requests.get("http://localhost:8080/v1/models", timeout=5).json()["data"][0]["id"]
        return ChatOpenAI(
            model=model_id,
            base_url="http://localhost:8080/v1",
            api_key="local",
            temperature=0,
        )

    elif provider == "local-ollama":
        # Ensure you have run: ollama pull <MODEL_NAME>
        model_name = os.getenv("MODEL_NAME", "qwen3-vl:8b")
        return ChatOllama(
            model=model_name,
            base_url="http://localhost:11434",
            temperature=0,
        )

    elif provider == "claude":
        # Requires ANTHROPIC_API_KEY in .env
        # Optionally set ANTHROPIC_MODEL (default: claude-sonnet-4-5)
        model_name = os.getenv("ANTHROPIC_MODEL", "claude-sonnet-4-5")
        return ChatAnthropic(
            model=model_name,
            api_key=os.getenv("ANTHROPIC_API_KEY"),
            temperature=0,
        )

    else:  # 'openai' or any other value falls back to OpenAI
        model_name = os.getenv("OPENAI_MODEL", "gpt-4o")
        return ChatOpenAI(
            model=model_name,
            api_key=os.getenv("OPENAI_API_KEY"),
            temperature=0,
        )


In [7]:
'''
# If you want to test your server/API
with client.chat.completions.create(
    model=model_id,  # Ensure you have run 'ollama pull llama3'
    messages=[{"role": "user", "content": "Explain why the sky is blue."}],
    stream=True,
) as response:
    for chunk in response:
        content = chunk.choices[0].delta.content
        if content:
            print(content, end="", flush=True)'''

In [ ]:
class Agent:
    """Base class for all pipeline agents.
    
    Provides shared functionality:
      - save_md: write markdown output to the stage's output directory
      - collect_files: walk a directory and read all files into dicts
      - calculate_total_size: sum character counts across a list of strings
      - invoke_llm: send a system + human message pair to an LLM
    """

    def __init__(self, stage, debug=False):
        self.stage = stage
        self.debug = debug
        self.system_prompt = ""

    def save_md(self, filename, data):
        """Write data to OUTPUT_DIR / self.stage / filename."""
        try:
            with open(OUTPUT_DIR / self.stage / filename, 'w') as f:
                f.write(data)
        except Exception as e:
            print(f"{e}: data")

    def collect_files(self, source_path):
        """Walk source_path and return all readable files as a list of dicts.
        
        Each dict has keys 'path' (relative to source_path) and 'content'.
        Hidden directories (starting with '.') are skipped.
        """
        all_files = []
        for root, _, files in os.walk(source_path):
            if '/.' in root:
                continue
            for file in files:
                full_path = os.path.join(root, file)
                try:
                    with open(full_path, 'r', encoding='utf-8') as f:
                        content = f.read()
                        all_files.append({
                            "path": os.path.relpath(full_path, source_path),
                            "content": content
                        })
                except Exception as e:
                    print(f"Could not read {full_path}: {e}")
        return all_files

    def calculate_total_size(self, items):
        """Sum the character lengths of a list of strings."""
        return sum(len(s) for s in items)

    def invoke_llm(self, llm, human_content):
        """Send self.system_prompt + human_content to the LLM and return the response text."""
        from langchain_core.messages import SystemMessage, HumanMessage
        messages = [
            SystemMessage(content=self.system_prompt),
            HumanMessage(content=human_content),
        ]
        response = llm.invoke(messages)
        return response.content


In [ ]:
# NOTE max_chars_per_chunk is very different depending on the setup...
# ~50k for a 4-bit quantized 8B model on an M4, ~500k for gpt-4o.
class FileSummarizer(Agent):
    """Stage 1 – walks a source repo, chunks files, and summarizes each chunk.
    
    File filtering uses TEXT_SUFFIXES from aai/repo_reader.py by default.
    Pass a custom list to `extensions` to override.
    """

    def __init__(self, repo_path, extensions=None, max_chars_per_chunk=50000, debug=False, stage=STAGES[0]):
        super().__init__(stage=stage, debug=debug)
        self.repo_path = repo_path
        self.extensions = extensions or list(TEXT_SUFFIXES)  # from aai/repo_reader.py
        self.max_chars_per_chunk = max_chars_per_chunk
        self.system_prompt = FILE_SUMMARIZER_PROMPT

    def _should_include(self, filename):
        """Return True if the file's extension is in self.extensions."""
        return any(filename.endswith(ext) for ext in self.extensions)

    def collect_files(self):
        """Walk self.repo_path and return only files matching self.extensions."""
        all_files = super().collect_files(self.repo_path)
        return [f for f in all_files if self._should_include(f['path'])]

    def create_chunks(self, all_files):
        """Group files into chunks that fit within the LLM character limit."""
        chunks = []
        current_chunk = []
        current_size = 0

        for file_data in all_files:
            file_size = len(file_data['content'])
            if current_size + file_size > self.max_chars_per_chunk and current_chunk:
                chunks.append(current_chunk)
                current_chunk = []
                current_size = 0
            current_chunk.append(file_data)
            current_size += file_size

        if current_chunk:
            chunks.append(current_chunk)
        return chunks

    # NOTE make async def if calling asynchronously
    def summarize_chunk(self, chunk, llm):
        """Send a chunk of files to the LLM and return the structured summary."""
        formatted_content = "\n\n".join([
            f"--- FILE: {f['path']} ---\n{f['content']}"
            for f in chunk
        ])
        return self.invoke_llm(llm, formatted_content)

    def process_all(self, chunks, llm):
        """Summarize every chunk and save each result to the stage output dir."""
        start = time.time()
        processed_sum = []
        for i, chunk in enumerate(chunks):
            print(f"Processing chunk {i}, with {len(chunk)} files")
            summary = self.summarize_chunk(chunk, llm)
            processed_sum.append(summary)
            self.save_md(f"summary{i}", summary)
            print(f"took {time.time() - start:.1f} seconds")
            start = time.time()
        return processed_sum


In [9]:
# Map files into chunks that can be summarized
llm = get_model()
scout = FileSummarizer("../code_base", debug=True)
files = scout.collect_files()
chunks = scout.create_chunks(files)

In [ ]:
scout.process_all(chunks, llm)

In [11]:
formatted_content = "\n\n".join([
    f"--- FILE: {f['path']} ---\n{f['content']}" 
    for f in chunks[0]
])

In [13]:
system_prompt = """
        You are a system archetect. Give a summary of the given code files.
        Output a Markdown response with a YAML frontmatter header.
        Identify:
        1. Module Name (based on folder or primary file)
        2. Key Dependencies (imports/includes)
        3. Primary Responsibilities
        4. Public APIs or Interfaces used
        """

In [14]:
# NOTE this timed out after 30m with ollama on a 16g M4 and a 8b qwen model after trying on zookeeper chunk 0
from langchain_core.messages import SystemMessage, HumanMessage
response = llm.invoke([
    SystemMessage(content=system_prompt),
    HumanMessage(content=formatted_content),
])

In [15]:
response

In [13]:
task = scout.summarize_chunk(chunk=chunks[0], llm=llm)

In [22]:
'''# Recommended way to run multiple chunks concurrently
tasks = [FileSummarizer.summarize_chunk(c) for c in chunks]
results = await asyncio.gather(*tasks)'''

In [128]:
chunks[0]

In [33]:
class ContextManager(Agent):
    """Stage 2 – reads scout summaries and recursively reduces them until they
    fit within the architect's context window.
    """

    def __init__(self, architect_threshold=20000, debug=False, stage=STAGES[1]):
        super().__init__(stage=stage, debug=debug)
        self.architect_threshold = architect_threshold
        self.scout_path = OUTPUT_DIR / STAGES[0]
        self.system_prompt = CONTEXT_MANAGER_PROMPT

    def collect_files(self):
        """Read all files from the scout output directory."""
        return super().collect_files(self.scout_path)

    def summarize_summaries(self, summary_batch, llm):
        """Collapse a group of summaries into a single 'Module Overview'."""
        combined_text = "\n\n".join(summary_batch)
        return self.invoke_llm(llm, combined_text)

    def reduce(self, summaries, llm):
        """Recursively shrink summaries until the total size fits the threshold."""
        current_summaries = summaries

        while self.calculate_total_size(current_summaries) > self.architect_threshold:
            print(f"Current size {len(str(current_summaries))} exceeds threshold. Reducing...")

            new_summaries = []
            chunk_size = 5
            for i in range(0, len(current_summaries), chunk_size):
                batch = current_summaries[i : i + chunk_size]
                reduced_summary = self.summarize_summaries(batch, llm)
                new_summaries.append(reduced_summary)
                self.save_md(f"reduced_sum{i}", reduced_summary)

            current_summaries = new_summaries

        if self.calculate_total_size(current_summaries) < self.architect_threshold:
            print(f"Current size {len(str(current_summaries))} does not need reduction")
            self.save_md("reduced_sum", str(current_summaries))
        return current_summaries


In [34]:
# Map files into chunks that can be summarized
aggregator = ContextManager(debug=True)
files = aggregator.collect_files()
reduced_files = aggregator.reduce(files, llm)

In [36]:
class ArchitectAgent(Agent):
    """Stage 3 – reads aggregated summaries and produces a Mermaid architecture diagram."""

    def __init__(self, debug=False, stage=STAGES[2]):
        super().__init__(stage=stage, debug=debug)
        self.agg_path = OUTPUT_DIR / STAGES[1]
        self.system_prompt = ARCHITECT_PROMPT

    def collect_files(self):
        """Read all files from the aggregator output directory."""
        return super().collect_files(self.agg_path)

    def summarize_summaries(self, summary_batch, llm):
        """Collapse a group of summaries into a Mermaid architecture diagram."""
        combined_text = "\n\n".join(summary_batch)
        return self.invoke_llm(llm, combined_text)
